# Análise de dados da plataforma de streaming

Este notebook inicia as análises exploratórias dos dados em `database/streaming.db`. As consultas são somente de leitura e novas análises podem ser adicionadas em seções próprias.

Se as bibliotecas ainda não estiverem instaladas no ambiente do Jupyter, execute no terminal do ambiente: `python -m install sqlalchemy pandas`.

## 1 — Importação das bibliotecas

SQLAlchemy gerencia a conexão com o SQLite e permite executar SQL; Pandas armazena os resultados em DataFrames. `Path` monta o caminho do banco de forma relativa ao diretório raiz do projeto.

In [3]:
pip install sqlalchemy pandas

   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---------------------------------------- 2.4/2.4 MB 24.1 MB/s  0:00:00
   ---------------------------------------- 0.0/10.9 MB ? eta -:--:--
   ---------------------------------------  10.7/10.9 MB 59.5 MB/s eta 0:00:01
   ---------------------------------------- 10.9/10.9 MB 47.8 MB/s  0:00:00
   ---------------------------------------- 0.0/12.7 MB ? eta -:--:--
   ------------------------------------- -- 12.1/12.7 MB 59.0 MB/s eta 0:00:01
   ---------------------------------------- 12.7/12.7 MB 43.2 MB/s  0:00:00

   ---------------------------------------- 0/3 [sqlalchemy]
   ---------------------------------------- 0/3 [sqlalchemy]
   ---------------------------------------- 0/3 [sqlalchemy]
   ---------------------------------------- 0/3 [sqlalchemy]
   ---------------------------------------- 0/3 [sqlalchemy]
   ---------------------------------------- 0/3 [sqlalchemy]
   -------------------------------------


[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
from pathlib import Path

import pandas as pd
from sqlalchemy import URL, create_engine, text

## 2 — Conexão com o banco de dados

O notebook espera ser iniciado a partir da raiz do projeto. O caminho é construído até `database/streaming.db`, onde o banco existente está localizado. A conexão de teste é aberta em um bloco `with` e fechada automaticamente.

In [5]:
DATABASE_PATH = Path.cwd() / "database" / "streaming.db"

if not DATABASE_PATH.is_file():
    raise FileNotFoundError(
        f"Banco não encontrado em {DATABASE_PATH}. Execute o notebook a partir da raiz do projeto."
    )

database_url = URL.create("sqlite", database=str(DATABASE_PATH))
engine = create_engine(database_url)

with engine.connect() as connection:
    connection.execute(text("SELECT 1")).scalar_one()

print(f"Conexão estabelecida: {DATABASE_PATH}")

Conexão estabelecida: d:\PROJETOS IA\streaming\database\streaming.db


## 3 — Análise de clientes Premium

Esta consulta seleciona os dados cadastrais de clientes cujo plano é `Premium`. O valor do plano é enviado como parâmetro SQL, e o Pandas guarda as linhas retornadas em `df_premium`. A verificação final compara o resultado com a contagem correspondente no banco e confirma que todas as linhas pertencem ao plano solicitado.

In [6]:
query_premium = text("""
    SELECT id, nome, data_adesao, tipo_plano, status_assinatura
    FROM clientes
    WHERE tipo_plano = :tipo_plano
    ORDER BY id
""")

with engine.connect() as connection:
    df_premium = pd.read_sql_query(
        query_premium,
        connection,
        params={"tipo_plano": "Premium"},
    )
    quantidade_premium_no_banco = connection.execute(
        text("SELECT COUNT(*) FROM clientes WHERE tipo_plano = :tipo_plano"),
        {"tipo_plano": "Premium"},
    ).scalar_one()

assert df_premium["tipo_plano"].eq("Premium").all()
assert len(df_premium) == quantidade_premium_no_banco
print(f"Clientes Premium encontrados: {len(df_premium)}")
display(df_premium)

Clientes Premium encontrados: 16


,id,nome,data_adesao,tipo_plano,status_assinatura
0,2,Bruno Moura,2022-02-10,Premium,Cancelado
1,6,Felipe Andrade,2022-07-20,Premium,Cancelado
2,7,Gabriela Carvalho,2022-08-29,Premium,Ativo
3,11,Karen Queiroz,2022-11-20,Premium,Ativo
4,15,Olivia Lima,2023-02-11,Premium,Ativo
5,16,Pedro Ferreira,2023-03-23,Premium,Ativo
6,20,Thiago Teixeira,2023-08-30,Premium,Ativo
7,24,Yasmin Barbosa,2023-11-21,Premium,Ativo
8,25,Caio Oliveira,2023-12-31,Premium,Ativo
9,29,Diego Freitas,2024-03-23,Premium,Ativo


### Insights de negócio

- Fatos observados: 16 dos 50 clientes (32%) têm plano Premium. A distribuição dos planos é próxima: Básico 17, Padrão 17 e Premium 16. Entre os clientes Premium, 11 estão ativos e 5 cancelados.
- Possível interpretação: o Premium representa uma parcela relevante, mas não dominante, desta amostra. A quantidade de clientes, por si só, não informa receita, margem ou valor percebido do plano.
- Possível recomendação: comparar receita e retenção por plano antes de alterar benefícios ou preços. Esta base fictícia e pequena não permite concluir que o plano cause maior permanência ou receita.

## Análise de consumo por gênero

Esta Análise soma os minutos assistidos em cada gênero para comparar o consumo das categorias. O `GROUP BY` reúne os eventos com o mesmo gênero e `SUM()` calcula o tempo total; assim, podemos identificar quais categorias concentram mais minutos assistidos.

In [7]:
query_genero = text("""
    SELECT genero, SUM(minutos_assistidos) AS total_minutos_assistidos
    FROM historico_visualizacao
    GROUP BY genero
    ORDER BY total_minutos_assistidos DESC, genero ASC
""")

with engine.connect() as connection:
    df_genero = pd.read_sql_query(query_genero, connection)
    generos_na_tabela = connection.execute(
        text("SELECT COUNT(DISTINCT genero) FROM historico_visualizacao")
    ).scalar_one()
    total_minutos_na_tabela = connection.execute(
        text("SELECT SUM(minutos_assistidos) FROM historico_visualizacao")
    ).scalar_one()

assert len(df_genero) == generos_na_tabela
assert df_genero["total_minutos_assistidos"].is_monotonic_decreasing
assert df_genero["total_minutos_assistidos"].sum() == total_minutos_na_tabela

display(df_genero)

,genero,total_minutos_assistidos
0,Suspense,466.0
1,Ação,442.0
2,Animação,415.0
3,Terror,410.0
4,Comédia,405.0
5,Documentário,400.0
6,Ficção Científica,316.0
7,Drama,311.0


### Insights de negócio

- Fatos observados: os 50 eventos somam 3.165 minutos. Suspense lidera com 466 minutos (14,7% do total), seguido de Ação (442) e Animação (415). Juntos, esses três gêneros representam 1.323 minutos (41,8%).
- Possível interpretação: neste conjunto, o consumo está distribuído entre vários gêneros, com vantagem moderada de Suspense.
- Possível recomendação: usar esses gêneros como hipótese para explorar curadoria e descoberta de conteúdo, acompanhando os resultados em uma base maior. Há apenas um evento por cliente nesta amostra, então os números descrevem estes registros e não estabelecem uma preferência estável dos usuários.

## Análise de clientes ativos sem visualizações recentes

A consulta parte de `clientes` e usa `LEFT JOIN` com `historico_visualizacao` pela relação `clientes.id = historico_visualizacao.id_cliente`. A condição do `LEFT JOIN` limita os registros agregados é visualizações anteriores ao início da janela de 30 dias; `AVG()` calcula a média individual desses minutos.

Um `NOT EXISTS` separado procura visualizações dentro dos últimos 30 dias e mantêm somente os clientes para os quais nenhuma foi encontrada. O `LEFT JOIN` preserva clientes sem histórico e, como `AVG()` não encontra valores para eles, a média fica `NULL`. Manter as condições do histórico no `ON` e no `NOT EXISTS`, em vez de filtrar essas colunas no `WHERE`, evita eliminar esses clientes sem atividade registrada. A janela usa `datetime('now', '-30 days')` do SQLite, portanto a data de referência ? dinâmica.

In [8]:
query_clientes_inativos = text("""
    SELECT
        c.id AS id_cliente,
        c.nome,
        c.tipo_plano,
        c.status_assinatura,
        AVG(h_antigo.minutos_assistidos) AS media_minutos_assistidos
    FROM clientes AS c
    LEFT JOIN historico_visualizacao AS h_antigo
        ON h_antigo.id_cliente = c.id
        AND datetime(h_antigo.data) < datetime('now', '-30 days')
    WHERE c.status_assinatura = :status_ativo
        AND NOT EXISTS (
            SELECT 1
            FROM historico_visualizacao AS h_recente
            WHERE h_recente.id_cliente = c.id
                AND datetime(h_recente.data) >= datetime('now', '-30 days')
                AND datetime(h_recente.data) <= datetime('now')
        )
    GROUP BY c.id, c.nome, c.tipo_plano, c.status_assinatura
    ORDER BY
        CASE WHEN media_minutos_assistidos IS NULL THEN 1 ELSE 0 END,
        media_minutos_assistidos DESC,
        c.nome ASC
""")

with engine.connect() as connection:
    df_clientes_inativos = pd.read_sql_query(
        query_clientes_inativos,
        connection,
        params={"status_ativo": "Ativo"},
    )

display(df_clientes_inativos)

,id_cliente,nome,tipo_plano,status_assinatura,media_minutos_assistidos
0,31,Felipe Ribeiro,Padrão,Ativo,118.0
1,40,Olivia Vargas,Padrão,Ativo,117.0
2,29,Diego Freitas,Premium,Ativo,113.0
3,27,Bruno Dias,Básico,Ativo,108.0
4,36,Karen Gomes,Básico,Ativo,107.0
5,16,Pedro Ferreira,Premium,Ativo,104.0
6,25,Caio Oliveira,Premium,Ativo,103.0
7,23,Wesley Vargas,Básico,Ativo,98.0
8,12,Lucas Freitas,Padrão,Ativo,94.0
9,21,Úrsula Almeida,Padrão,Ativo,93.0


### Insights de negócio

- Fatos observados: 37 dos 37 clientes ativos aparecem sem visualizações nos últimos 30 dias. O evento mais recente da base é de 29/11/2025. Cada cliente tem somente um evento registrado; portanto, a média histórica exibida para cada um equivale a esse único evento. Não há clientes sem histórico neste conjunto atual.
- Possível interpretação: em uma operação real, ausência recente de atividade poderia sinalizar baixo engajamento. Aqui, como a base fictícia não tem eventos recentes e é pequena, o resultado também evidencia a falta de atualização dos dados e não comprova risco real de cancelamento.
- Possível recomendação: antes de iniciar uma ação de retenção baseada neste resultado, atualizar e validar a coleta de eventos. Com histórico mais completo, a média individual poderia representar melhor o comportamento de cada cliente.